In [1]:
import pandas as pd
import numpy as np

orders = pd.read_csv('../data/processed/orders_delivered.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
order_items = pd.read_csv('../data/processed/order_items_clean.csv')
customers = pd.read_csv('../data/processed/customers_clean.csv', dtype={'customer_zip_code_prefix': str})
payments = pd.read_csv('../data/processed/payments_clean.csv')

In [3]:
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days

orders['is_late'] = (orders['delivery_delay_days'] > 0).astype(int)

orders['delivery_delay_days'].describe()

count    96470.000000
mean       -11.875889
std         10.182105
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

In [4]:
orders['is_late'].value_counts(normalize=True)

is_late
0    0.932269
1    0.067731
Name: proportion, dtype: float64

In [5]:
orders['order_year'] = orders['order_purchase_timestamp'].dt.year
orders['order_month'] = orders['order_purchase_timestamp'].dt.month
orders['order_year_month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

orders[['order_purchase_timestamp', 'order_year', 'order_month', 'order_year_month']].head()

,order_purchase_timestamp,order_year,order_month,order_year_month
0,2017-10-02 10:56:33,2017,10,2017-10
1,2018-07-24 20:41:37,2018,7,2018-07
2,2018-08-08 08:38:49,2018,8,2018-08
3,2017-11-18 19:28:06,2017,11,2017-11
4,2018-02-13 21:18:39,2018,2,2018-02


In [6]:
order_items_agg = order_items.groupby('order_id').agg(
    items_price=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    num_items=('order_item_id', 'count')
).reset_index()

order_items_agg['order_value'] = order_items_agg['items_price'] + order_items_agg['freight_value']
order_items_agg.head()

,order_id,items_price,freight_value,num_items,order_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,218.04


In [7]:
orders = orders.merge(order_items_agg, on='order_id', how='left')
orders['order_value'].isnull().sum()

np.int64(0)

In [8]:
orders['order_value'].describe()

count    96470.000000
mean       159.825799
std        218.802007
min          9.590000
25%         61.850000
50%        105.280000
75%        176.260000
max      13664.080000
Name: order_value, dtype: float64

In [9]:
orders_cust = orders.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']],
    on='customer_id', how='left'
)

In [10]:
snapshot_date = orders_cust['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = orders_cust.groupby('customer_unique_id').agg(
    recency_days=('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('order_value', 'sum')
).reset_index()

rfm.describe()

,recency_days,frequency,monetary
count,93350.000000,93350.000000,93350.000000
mean,237.950070,1.033423,165.167593
std,152.589932,0.209106,226.300536
min,1.000000,1.000000,9.590000
25%,114.000000,1.000000,63.010000
50%,219.000000,1.000000,107.780000
75%,346.000000,1.000000,182.505000
max,714.000000,15.000000,13664.080000


In [11]:
(rfm['frequency'] == 1).mean()

np.float64(0.9699946438136047)

In [12]:
orders.to_csv('../data/processed/orders_features.csv', index=False)
rfm.to_csv('../data/processed/customer_rfm.csv', index=False)